# 01 — Data Exploration

**Dissertation: Phishing Detection Pipeline**

This notebook performs exploratory data analysis (EDA) on the three phishing/spam datasets used in this project:

| Dataset | Type | Size (approx.) | Source |
|---------|------|----------------|--------|
| SpamAssassin Public Corpus | Email | ~6 000 messages | spamassassin.apache.org |
| Nazario Phishing Corpus | Email | ~2 000 messages | monkey.org/~jose/phishing/ |
| SMS Spam Collection | SMS | ~5 574 messages | UCI ML Repository |

**Objectives**
1. Verify data loading and preprocessing.
2. Examine class balance.
3. Analyse message length distributions.
4. Inspect URL count distributions.
5. Explore top tokens and urgency-word usage.
6. Visualise feature correlations.


## 0. Imports and configuration

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from collections import Counter

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

# Make src importable from the notebook directory
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.text_pipeline.data_loader import (
    load_all,
    load_spamassassin,
    load_nazario,
    load_sms_spam_collection,
    preprocess_dataframe,
)
from src.text_pipeline.feature_extractor import (
    extract_text_features,
    extract_url_features,
    aggregate_url_features,
    URLFeatureTransformer,
    TextStatFeatureTransformer,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIGSIZE = (12, 4)
RANDOM_STATE = 42

# ── Adjust these paths to where you stored the raw data ──────────────────────
DATA_DIR = PROJECT_ROOT / "data" / "raw"

SPAMASSASSIN_SPAM = DATA_DIR / "spam"
SPAMASSASSIN_HAM  = DATA_DIR / "easy_ham"
NAZARIO_PATH      = DATA_DIR / "nazario"
SMS_PATH          = DATA_DIR / "SMSSpamCollection"
# ─────────────────────────────────────────────────────────────────────────────

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)

## 1. Load datasets

Each dataset is loaded individually so we can inspect them, then combined into a single frame.

In [ ]:
dfs = {}

# ── SpamAssassin ──────────────────────────────────────────────────────────────
if SPAMASSASSIN_SPAM.exists() and SPAMASSASSIN_HAM.exists():
    dfs["SpamAssassin"] = load_spamassassin(SPAMASSASSIN_SPAM, SPAMASSASSIN_HAM)
    print(f"SpamAssassin: {len(dfs['SpamAssassin'])} messages loaded")
else:
    print("SpamAssassin data not found — skipping.")

# ── Nazario ───────────────────────────────────────────────────────────────────
if NAZARIO_PATH.exists():
    dfs["Nazario"] = load_nazario(NAZARIO_PATH)
    print(f"Nazario: {len(dfs['Nazario'])} messages loaded")
else:
    print("Nazario data not found — skipping.")

# ── SMS Spam Collection ───────────────────────────────────────────────────────
if SMS_PATH.exists():
    dfs["SMS"] = load_sms_spam_collection(SMS_PATH)
    print(f"SMS Spam Collection: {len(dfs['SMS'])} messages loaded")
else:
    print("SMS Spam Collection data not found — skipping.")

if not dfs:
    raise RuntimeError(
        "No datasets found. Please download the raw data into data/raw/ "
        "and update the path variables in cell 0."
    )

# Combine and preprocess
df_raw = pd.concat(dfs.values(), ignore_index=True)
print(f"\nCombined (before preprocessing): {len(df_raw)} messages")
df = preprocess_dataframe(df_raw, remove_urls=False, drop_duplicates=True)
print(f"Combined (after preprocessing): {len(df)} messages")

In [ ]:
df.head(3)

In [ ]:
print("Schema:")
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())

## 2. Class balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# Overall
label_counts = df["label"].map({0: "Legitimate", 1: "Phishing/Spam"}).value_counts()
axes[0].pie(
    label_counts,
    labels=label_counts.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("muted", 2),
    startangle=90,
)
axes[0].set_title("Overall class balance")

# Per-source
src_label = df.groupby(["source", "label"]).size().unstack(fill_value=0)
src_label.columns = ["Legitimate", "Phishing/Spam"]
src_label.plot(kind="bar", ax=axes[1], color=sns.color_palette("muted", 2))
axes[1].set_title("Class balance per dataset")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "class_balance.png", dpi=150, bbox_inches="tight")
plt.show()

print(df.groupby(["source", "label"]).size().rename("count").reset_index())

## 3. Message length distributions

In [ ]:
df["text_length"] = df["text"].str.len()
df["word_count"]  = df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

for ax, col, xlabel in zip(
    axes,
    ["text_length", "word_count"],
    ["Character count", "Word count"],
):
    for label_val, label_name, color in [
        (0, "Legitimate", "steelblue"),
        (1, "Phishing/Spam", "salmon"),
    ]:
        subset = df[df["label"] == label_val][col].clip(upper=df[col].quantile(0.99))
        sns.histplot(subset, ax=ax, label=label_name, color=color, bins=60, alpha=0.55, kde=True)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Count")
    ax.set_title(f"{xlabel} distribution")
    ax.legend()

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "length_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

print(df.groupby("label")[["text_length", "word_count"]].describe().round(1))

## 4. URL count distribution

In [ ]:
df["url_count"] = df["urls"].apply(len)

fig, axes = plt.subplots(1, 2, figsize=FIGSIZE)

# Histogram
for label_val, label_name, color in [
    (0, "Legitimate", "steelblue"),
    (1, "Phishing/Spam", "salmon"),
]:
    subset = df[df["label"] == label_val]["url_count"].clip(upper=20)
    sns.histplot(subset, ax=axes[0], label=label_name, color=color, bins=21, alpha=0.55)
axes[0].set_title("URL count per message")
axes[0].set_xlabel("Number of URLs (capped at 20)")
axes[0].legend()

# Box plot
url_box = df[["label", "url_count"]].copy()
url_box["label"] = url_box["label"].map({0: "Legitimate", 1: "Phishing/Spam"})
sns.boxplot(data=url_box, x="label", y="url_count", palette="muted", ax=axes[1])
axes[1].set_ylim(0, df["url_count"].quantile(0.99) + 1)
axes[1].set_title("URL count box plot")
axes[1].set_xlabel("")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "url_count_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print(df.groupby("label")["url_count"].describe().round(2))

## 5. Top tokens and urgency word analysis

In [ ]:
STOP = set(stopwords.words("english"))

def top_tokens(texts, n=20, extra_stop=None):
    """Return the *n* most common non-stopword tokens from a corpus."""
    stop = STOP | (extra_stop or set())
    tokens = [
        tok
        for text in texts
        for tok in word_tokenize(str(text).lower())
        if tok.isalpha() and tok not in stop and len(tok) > 2
    ]
    return Counter(tokens).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label_val, title, color in [
    (axes[0], 0, "Top 20 tokens — Legitimate", "steelblue"),
    (axes[1], 1, "Top 20 tokens — Phishing/Spam", "salmon"),
]:
    subset = df[df["label"] == label_val]["text"]
    counts = top_tokens(subset, n=20)
    words, freqs = zip(*counts)
    ax.barh(words[::-1], freqs[::-1], color=color, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("Frequency")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "top_tokens.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Urgency word usage rate
from src.text_pipeline.feature_extractor import _URGENCY_PHRASES

def urgency_rate(texts):
    hits = sum(
        any(p in str(t).lower() for p in _URGENCY_PHRASES)
        for t in texts
    )
    return hits / max(len(texts), 1)

print("Urgency phrase hit rate:")
for label_val, name in [(0, "Legitimate"), (1, "Phishing/Spam")]:
    rate = urgency_rate(df[df["label"] == label_val]["text"])
    print(f"  {name}: {rate:.1%}")

## 6. Hand-crafted feature distributions

In [ ]:
# Build statistical text feature matrix for a sample (full corpus can be slow)
SAMPLE_SIZE = min(2000, len(df))
df_sample = df.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)

text_feats = TextStatFeatureTransformer().fit_transform(df_sample)
feat_names  = TextStatFeatureTransformer().fit(df_sample).get_feature_names_out()
df_text_feats = pd.DataFrame(text_feats, columns=feat_names)
df_text_feats["label"] = df_sample["label"].values

df_text_feats.groupby("label")[feat_names].mean().T.rename(
    columns={0: "Legitimate", 1: "Phishing/Spam"}
).style.background_gradient(axis=1, cmap="RdYlGn_r")

In [ ]:
# Select a handful of discriminative features to visualise
VIS_FEATS = [
    "urgency_word_count",
    "exclamation_count",
    "uppercase_ratio",
    "html_tag_count",
    "avg_word_length",
    "lexical_diversity",
]
VIS_FEATS = [f for f in VIS_FEATS if f in feat_names]

n_cols = 3
n_rows = math.ceil(len(VIS_FEATS) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4 * n_rows))
axes = axes.flatten()

import math
for i, feat in enumerate(VIS_FEATS):
    for label_val, label_name, color in [
        (0, "Legitimate", "steelblue"),
        (1, "Phishing/Spam", "salmon"),
    ]:
        vals = df_text_feats[df_text_feats["label"] == label_val][feat]
        sns.histplot(vals.clip(upper=vals.quantile(0.99)), ax=axes[i],
                     label=label_name, color=color, bins=40, alpha=0.55, kde=True)
    axes[i].set_title(feat.replace("_", " ").title())
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Correlation heatmap of text features

In [ ]:
corr_df = df_text_feats[list(feat_names)].corr()

plt.figure(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(
    corr_df,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.4,
    vmin=-1,
    vmax=1,
)
plt.title("Text feature correlation matrix")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "feature_correlation.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. URL feature spot-check

In [ ]:
# Compare aggregate URL features between phishing and legitimate messages
url_feats = URLFeatureTransformer().fit_transform(df_sample)
url_feat_names = URLFeatureTransformer().fit(df_sample).get_feature_names_out()
df_url_feats = pd.DataFrame(url_feats, columns=url_feat_names)
df_url_feats["label"] = df_sample["label"].values

mean_comparison = df_url_feats.groupby("label")[url_feat_names].mean().T
mean_comparison.columns = ["Legitimate", "Phishing/Spam"]
mean_comparison["ratio (P/L)"] = (
    mean_comparison["Phishing/Spam"] / mean_comparison["Legitimate"].replace(0, np.nan)
).round(2)
mean_comparison.sort_values("ratio (P/L)", ascending=False).head(15)

## 9. Save cleaned dataset for downstream notebooks

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Serialise url lists as pipe-delimited strings for CSV compatibility
df_save = df.copy()
df_save["urls"] = df_save["urls"].apply(lambda x: "|".join(x) if isinstance(x, list) else "")
out_path = processed_dir / "combined_dataset.csv"
df_save.to_csv(out_path, index=False)
print(f"Saved {len(df_save)} rows → {out_path}")

## 10. Summary

| Observation | Finding |
|---|---|
| Class balance | Check the pie chart above — imbalance may require SMOTE or class-weight adjustment |
| Message length | Phishing emails tend to be shorter; SMS phishing shorter still |
| URL count | Phishing messages contain significantly more URLs on average |
| Urgency language | Phishing/spam messages have a higher urgency-phrase hit rate |
| HTML tags | Phishing emails often embed more HTML (obfuscation) |
| Correlated features | `char_count` / `word_count` are highly correlated — consider dropping one |

**Next steps**: `02_feature_engineering.ipynb` — build full feature matrices and run baseline classifiers.